# Phase 1 数据探索

## Synthetic Data 声明

本 Notebook 只读分析程序生成的 **Synthetic E-commerce Data / 模拟电商数据**，不代表真实企业、真实用户或真实经营表现。Notebook 不构建、不修改、不覆盖任何数据快照。

## 数据范围

分析对象为五张业务表：`products`、`customers`、`traffic`、`marketing` 和 `orders`。下方代码会从当前工作目录向上定位项目根目录，避免依赖 Notebook 的启动位置。

若快照不存在，请先在项目根目录运行 `make PYTHON=python3.12 phase1-data`（也可将 `PYTHON` 替换为其他 Python 3.11+ 可执行文件），再执行本 Notebook。

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    candidates = (Path.cwd(), *Path.cwd().parents)
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "app").is_dir():
            return candidate
    raise FileNotFoundError(
        "未找到项目根目录；请从仓库内启动 Jupyter，并先运行 make phase1-data。"
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

open_dataset = importlib.import_module("app.data.database").open_dataset


SNAPSHOTS = {
    "Development": PROJECT_ROOT / "data/synthetic/v1",
    "Public Validation": PROJECT_ROOT / "data/synthetic/public-validation-v1",
}

missing = [name for name, path in SNAPSHOTS.items() if not path.is_dir()]
if missing:
    raise FileNotFoundError(
        f"缺少数据快照 {missing}；请先在项目根目录运行 make phase1-data。"
    )

catalogs = {"Development": open_dataset(SNAPSHOTS["Development"])}
with open_dataset(SNAPSHOTS["Public Validation"]) as public_validation_catalog:
    public_validation_summary = dict(public_validation_catalog.verified_summary)
del public_validation_catalog

In [ ]:
development_catalog = catalogs["Development"]
development_summary = development_catalog.verified_summary
start_date, end_date = development_catalog.execute(
    "select min(date), max(date) from traffic"
).fetchone()
summary_rows = [
    {
        "dataset": "Development",
        "source_label": development_summary["source_label"],
        "dataset_version": development_summary["dataset_version"],
        "dataset_id": development_summary["dataset_id"],
        "start_date": start_date,
        "end_date": end_date,
        **{
            f"rows_{table}": rows
            for table, rows in development_summary["row_counts"].items()
        },
        "quality_status": f"{development_summary['quality_status']} (verified)",
    },
    {
        "dataset": "Public Validation",
        "source_label": public_validation_summary["source_label"],
        "dataset_version": public_validation_summary["dataset_version"],
        "dataset_id": public_validation_summary["dataset_id"],
        "start_date": None,
        "end_date": None,
        **{
            f"rows_{table}": rows
            for table, rows in public_validation_summary["row_counts"].items()
        },
        "quality_status": f"{public_validation_summary['quality_status']} (verified)",
    },
]

dataset_summary = pd.DataFrame(summary_rows)
dataset_summary

## 质量检查

仅展示通过 `open_dataset` 完成 Manifest、Parquet 哈希、逻辑内容、行数及质量报告交叉校验后的状态；Notebook 不直接读取或展示未验证的元数据。

In [ ]:
quality_summary = dataset_summary[
    [
        "dataset",
        "quality_status",
        "rows_products",
        "rows_customers",
        "rows_traffic",
        "rows_marketing",
        "rows_orders",
    ]
]
quality_summary

## 指标分布

仅对 Development 快照按日聚合并描述 **GMV**、**Orders**、**AOV**、**CTR**、**CVR**、**Refund Rate** 和 **ROAS**。`coverage` 是当日有观测值的流量行占比；只要当日覆盖不完整，CTR 与 CVR 就保持 `null`，不把缺失误当作零。

In [ ]:
daily_metrics = catalogs["Development"].execute(
    """
    with traffic_daily as (
        select
            date,
            count(*) as total_rows,
            count(*) filter (where not is_missing) as observed_rows,
            sum(impressions) filter (where not is_missing) as impressions,
            sum(clicks) filter (where not is_missing) as clicks,
            sum(visits) filter (where not is_missing) as visits
        from traffic
        group by date
    ),
    orders_daily as (
        select
            order_date as date,
            sum(revenue) as gmv,
            count(distinct order_id) as orders,
            sum(case when is_refund then 1 else 0 end) as refunds
        from orders
        group by order_date
    ),
    marketing_daily as (
        select date, sum(spend) as spend
        from marketing
        group by date
    )
    select
        t.date,
        t.observed_rows,
        t.total_rows,
        t.observed_rows::double / nullif(t.total_rows, 0) as coverage,
        o.gmv as "GMV",
        o.orders as "Orders",
        o.gmv / nullif(o.orders, 0) as "AOV",
        case when t.observed_rows = t.total_rows
            then t.clicks::double / nullif(t.impressions, 0)
        end as "CTR",
        case when t.observed_rows = t.total_rows
            then o.orders::double / nullif(t.visits, 0)
        end as "CVR",
        o.refunds::double / nullif(o.orders, 0) as "Refund Rate",
        o.gmv / nullif(m.spend, 0) as "ROAS"
    from traffic_daily t
    left join orders_daily o using (date)
    left join marketing_daily m using (date)
    order by t.date
    """
).fetch_df()

metric_columns = ["GMV", "Orders", "AOV", "CTR", "CVR", "Refund Rate", "ROAS"]
daily_metrics[["coverage", *metric_columns]].describe().transpose()

## 异常场景

为避免暴露预设答案和异常配置，这里不加载场景配置，也不展示目标商品或强度参数。仅展示 Development 中从已验证事实表观察到的日期级覆盖缺口与聚合指标。

In [ ]:
coverage_gaps = daily_metrics.loc[
    daily_metrics["coverage"].lt(1),
    ["date", "observed_rows", "total_rows", "coverage", "CTR", "CVR"],
]
coverage_gaps

In [ ]:
daily_metrics.loc[
    daily_metrics["coverage"].eq(1),
    ["date", "coverage", "GMV", "Orders", "CTR", "CVR", "Refund Rate", "ROAS"],
].describe().transpose()

## 双数据集说明

- **Development** 使用 `data/synthetic/v1`，用于开发期分析与验证。
- **Public Validation** 使用隔离快照 `data/synthetic/public-validation-v1`；Notebook 仅通过局部安全 Catalog 复制经验证的身份、行数与质量状态，随后立即关闭并删除句柄，不加载生成配置，也不查询或展示业务指标。
- 两套快照分别拥有 Dataset ID。Notebook 不合并身份，也不把 Development 的结论外推到 Public Validation。

In [ ]:
dataset_summary

## 局限性

- 数据由固定规则生成，只用于受控评测，不代表真实电商生产分布。
- 预设场景有限，不能覆盖开放世界中的全部异常。
- 描述统计只能说明当前快照中的模式，不能证明因果关系或生产泛化能力。
- Public Validation 虽与 Development 独立生成，但仍共享同一生成器机制，分布差异有限；它是公开验证集，不是盲测。
- 本阶段未运行 Agent；所有 Agent Evaluation 结果均为 **Pending / Not Run**，因此这里不报告任何 Agent 效果数字。
- Notebook 不保存执行输出；查看结果时应在本地基于已验证快照重新运行。